<a href="https://colab.research.google.com/github/praju-007anchan/IN126060102_NLP/blob/main/Copy_of_BERT_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import nbformat

# Load notebook
nb = nbformat.read("your_notebook.ipynb", as_version=4)

# Remove broken widgets
if 'widgets' in nb['metadata']:
    del nb['metadata']['widgets']

# Save fixed notebook
nbformat.write(nb, "your_notebook_fixed.ipynb")
print("Notebook fixed!")

In [ ]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [ ]:
!pip install transformers datasets -q

import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import numpy as np
import torch
import torch.nn as nn

from datasets import load_dataset
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

from transformers import AutoModel, BertTokenizerFast
from torch.optim import AdamW # Corrected import to use AdamW from torch.optim

from torch.utils.data import DataLoader, TensorDataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [ ]:
dataset = load_dataset("imdb")

train_data = dataset["train"]
test_data = dataset["test"]

print(train_data[0])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far be

In [ ]:
train_data = train_data.shuffle(seed=42).select(range(5000))
test_data = test_data.shuffle(seed=42).select(range(2000))

In [ ]:
from sklearn.model_selection import train_test_split

texts = [x['text'] for x in train_data]
labels = [x['label'] for x in train_data]

train_text, val_text, train_labels, val_labels = train_test_split(
    texts, labels, test_size=0.1, stratify=labels, random_state=42
)

test_text = [x['text'] for x in test_data]
test_labels = [x['label'] for x in test_data]

In [ ]:
tokenizer = BertTokenizerFast.from_pretrained('bert-base-uncased')

max_len = 128

tokens_train = tokenizer(
    train_text,
    max_length=max_len,
    padding=True,
    truncation=True
)

tokens_val = tokenizer(
    val_text,
    max_length=max_len,
    padding=True,
    truncation=True
)

tokens_test = tokenizer(
    test_text,
    max_length=max_len,
    padding=True,
    truncation=True
)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
train_seq = torch.tensor(tokens_train['input_ids'])
train_mask = torch.tensor(tokens_train['attention_mask'])
train_y = torch.tensor(train_labels)

val_seq = torch.tensor(tokens_val['input_ids'])
val_mask = torch.tensor(tokens_val['attention_mask'])
val_y = torch.tensor(val_labels)

test_seq = torch.tensor(tokens_test['input_ids'])
test_mask = torch.tensor(tokens_test['attention_mask'])
test_y = torch.tensor(test_labels)

In [ ]:
batch_size = 16

train_loader = DataLoader(TensorDataset(train_seq, train_mask, train_y), batch_size=batch_size, shuffle=True)
val_loader = DataLoader(TensorDataset(val_seq, val_mask, val_y), batch_size=batch_size)

In [ ]:
for param in bert.parameters():
    param.requires_grad = False

In [ ]:
bert = AutoModel.from_pretrained('bert-base-uncased')

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
class BERT_Arch(nn.Module):
    def __init__(self, bert):
        super().__init__()
        self.bert = bert
        self.fc1 = nn.Linear(768,256)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.1)
        self.fc2 = nn.Linear(256,2)
        self.logsoftmax = nn.LogSoftmax(dim=1)

    def forward(self, sent_id, mask):
        _, cls = self.bert(sent_id, attention_mask=mask, return_dict=False)
        x = self.fc1(cls)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return self.logsoftmax(x)

model = BERT_Arch(bert).to(device)

In [ ]:
optimizer = AdamW(model.parameters(), lr=2e-5)

class_wts = compute_class_weight(class_weight='balanced', classes=np.unique(train_labels), y=train_labels)
weights = torch.tensor(class_wts, dtype=torch.float).to(device)

loss_fn = nn.NLLLoss(weight=weights)

In [ ]:
def train():
    model.train()
    total_loss = 0

    for batch in train_loader:
        batch = [b.to(device) for b in batch]
        sent_id, mask, labels = batch

        model.zero_grad()
        preds = model(sent_id, mask)
        loss = loss_fn(preds, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(train_loader)

In [ ]:
def evaluate():
    model.eval()
    total_loss = 0
    preds_all = []

    with torch.no_grad():
        for batch in val_loader:
            batch = [b.to(device) for b in batch]
            sent_id, mask, labels = batch

            preds = model(sent_id, mask)
            loss = loss_fn(preds, labels)

            total_loss += loss.item()
            preds_all.append(preds.cpu().numpy())

    return total_loss / len(val_loader), np.concatenate(preds_all)

In [ ]:
import torch
print(torch.cuda.is_available())

True


In [ ]:
epochs = 2

for epoch in range(epochs):
    print(f"\nEpoch {epoch+1}")

    train_loss = train()
    val_loss, _ = evaluate()

    print("Train Loss:", round(train_loss,3))
    print("Val Loss:", round(val_loss,3))


Epoch 1
Train Loss: 0.685
Val Loss: 0.688

Epoch 2
Train Loss: 0.668
Val Loss: 0.677


In [ ]:
model.eval()

with torch.no_grad():
    preds = model(test_seq.to(device), test_mask.to(device))
    preds = preds.cpu().numpy()

preds = np.argmax(preds, axis=1)

print(classification_report(test_y, preds))

              precision    recall  f1-score   support

           0       0.70      0.43      0.53      1000
           1       0.59      0.82      0.69      1000

    accuracy                           0.62      2000
   macro avg       0.65      0.62      0.61      2000
weighted avg       0.65      0.62      0.61      2000



In [ ]:
cm = confusion_matrix(test_y, preds)
print(cm)

[[430 570]
 [180 820]]
